# Create NBCF Awards (National Breast Cancer Foundation, Australia)

Creates National Breast Cancer Foundation (NBCF) awards from the research
projects NBCF publishes on its own site:

- current projects: https://nbcf.org.au/research/research-projects/ (post type `project`, 140)
- completed projects: https://nbcf.org.au/research/research-projects-completed/ (post type `project-completed`, 100)

Projects are enumerated from the Yoast sitemaps. Each project page gives the
title, a person card (institution, position, lead researcher) and a free-text
description; the funding year and grant type come from the site's listing
filters (static archive URLs `.../year/{yyyy}/` and `.../grant/{type}/`), and the
research category (Prevent / Detect / Treat / Stop / Quality Of Life) from the
listing cards. **240 projects, award years 2015-2026, 100% title/description/
year/grant type, 99.6% lead researcher + institution** (local run 2026-10-01).

NBCF publishes **no grant number and no per-project amount** on the site:
amount/currency are NULL (§6.7 amount check waived — the funder does not
publish amounts), `start_date` is 1 January of the listed award year
(year-only precedent), `end_date` NULL. The site carries NBCF's current and
completed project portfolio only (~240 projects since 2015); NBCF's older
grants are not listed.

**Scope:** every listed project is kept. One 2026 "Other" grant ("Transforming
prevention and early detection – A new world-leading centre for high-risk breast
cancer") funds MRI equipment for a research centre; kept and flagged. Its person
card is an image caption, so it ships with no lead investigator.

**Prerequisites:**
- Run `scripts/local/nbcf_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/nbcf/nbcf_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** synthetic and stable, `NBCF-{WordPress slug}`.
Works cite NBCF's scheme codes (`IIRS-19-016`, `PF-16-011`, `ECF-17-002`, ...:
827 of the 1,371 stub rows on F4320320438), but NBCF does not print them on the
project pages, so 0 stubs collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320438`
- display_name: National Breast Cancer Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 0120ky124, doi 10.13039/501100001026)

**Priority:** `566` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nbcf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nbcf/nbcf_projects.parquet`;

In [ ]:
%sql
SELECT post_type, COUNT(*) as projects FROM openalex.awards.nbcf_raw GROUP BY post_type;

In [ ]:
%sql
DESCRIBE openalex.awards.nbcf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nbcf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320438 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320438;

## Step 2: Create NBCF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nbcf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320438  -- National Breast Cancer Foundation
),

g AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS yr,
        NULLIF(TRIM(lead_family_name), '') AS family
    FROM openalex.awards.nbcf_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.grant_type = 'other' THEN 'grant' ELSE 'research' END as funding_type,
    CASE g.grant_type
        WHEN 'research-project-grant' THEN 'Research Project Grant'
        WHEN 'pink-sky-grants' THEN 'Pink Sky Grant'
        WHEN 'collaborative-research-accelerator-grant' THEN 'Collaborative Research Accelerator Grant'
        WHEN 'other' THEN 'Other'
        ELSE NULLIF(TRIM(g.grant_type), '')
    END as funder_scheme,
    'nbcf_research_projects' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.award_year, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nbcf_research_projects' AND priority = 566;

-- Insert into openalex_awards_raw with priority.
-- Priority 566: direct-from-funder ingest of NBCF's own project pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    566 as priority
FROM openalex.awards.nbcf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids, COUNT(DISTINCT funder_award_id) as distinct_award_ids
FROM openalex.awards.nbcf_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is the synthetic NBCF-{slug} form.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^NBCF-[a-z0-9%-]+$' THEN 1 ELSE 0 END) as well_formed
FROM openalex.awards.nbcf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.nbcf_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.nbcf_awards
GROUP BY start_year, funder_scheme ORDER BY start_year, funder_scheme;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.nbcf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.nbcf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Coverage (expect 100% title/description/start year, ~99.6% PI and institution, no amounts published).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.nbcf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nbcf_research_projects'
GROUP BY provenance, priority;